In [ ]:
!pip -q install transformers biopython scikit-learn matplotlib pandas requests

In [ ]:
import io
import math
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from Bio import SeqIO
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_similarity, euclidean_distances

import torch
from transformers import EsmModel, EsmTokenizer


In [ ]:
def fetch_uniprot_fasta(uniprot_id: str):
    url = f"https://rest.uniprot.org/uniprotkb/{uniprot_id}.fasta"
    r = requests.get(url, timeout=30)
    r.raise_for_status()
    record = next(SeqIO.parse(io.StringIO(r.text), "fasta"))
    return str(record.seq), record.description

tp53_seq, tp53_desc = fetch_uniprot_fasta("P04637")

print(tp53_desc)
print("Length:", len(tp53_seq))
print(tp53_seq[:80] + "...")

In [ ]:
# Actualización de nombres de grupos para que sean más descriptivos
position_groups = {
    "Hotspots canónicos": [175, 220, 245, 248],
    "Estructurales": [176, 179, 249, 282],
    "Comparativos del DBD": [125, 138, 151, 157],
}

all_positions = []
for group, pos_list in position_groups.items():
    for p in pos_list:
        all_positions.append((p, group))

amino_acids = list("ACDEFGHIKLMNPQRSTVWY")

print("Grupos actualizados:", position_groups.keys())

In [ ]:
for pos, group in all_positions:
    wt_aa = tp53_seq[pos - 1]
    print(f"{group:15s} {wt_aa}{pos}")

In [ ]:
def generate_single_mutants_with_metadata(seq, positions_with_groups, amino_acids):
    mutants = []
    seq_list = list(seq)

    for pos_1based, group in positions_with_groups:
        idx = pos_1based - 1
        wt_aa = seq_list[idx]

        for aa in amino_acids:
            if aa == wt_aa:
                continue

            mut_seq = seq_list.copy()
            mut_seq[idx] = aa
            mut_seq = "".join(mut_seq)

            mutants.append({
                "mutant": f"{wt_aa}{pos_1based}{aa}",
                "position": pos_1based,
                "wt_aa": wt_aa,
                "mut_aa": aa,
                "group": group,
                "sequence": mut_seq,
            })

    return mutants

mutants = generate_single_mutants_with_metadata(tp53_seq, all_positions, amino_acids)

print("Number of mutants:", len(mutants))
pd.DataFrame(mutants).head()

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

model_name = "facebook/esm2_t33_650M_UR50D"
tokenizer = EsmTokenizer.from_pretrained(model_name)
model = EsmModel.from_pretrained(model_name)

model.eval()
model = model.to(device)


In [ ]:
def get_residue_representations_batched(
    seqs_with_names,
    model,
    tokenizer,
    device="cpu",
    repr_layers=None,   # None = last_hidden_state, o lista como [33]
    batch_size=16
):
    """Extrae representaciones por residuo usando el modelo ESM de Hugging Face.

    Args:
        repr_layers: None para usar solo last_hidden_state,
                     o lista de índices de capa (1..num_layers) para extraer
                     múltiples capas con output_hidden_states=True.
    Returns:
        labels: list[str]
        all_residue_reps: list[np.ndarray] si repr_layers es None/una capa;
                         dict[int, list[np.ndarray]] si múltiples capas.
    """
    all_labels = []
    multi_layer = repr_layers is not None and len(repr_layers) > 1
    all_residue_reps = {} if multi_layer else []

    output_hidden = repr_layers is not None

    for start in range(0, len(seqs_with_names), batch_size):
        batch = seqs_with_names[start:start + batch_size]
        batch_labels = [name for name, seq in batch]
        batch_strs = [seq for name, seq in batch]

        inputs = tokenizer(
            batch_strs, return_tensors="pt", padding=True, truncation=True
        )
        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = model(**inputs, output_hidden_states=output_hidden)

        for i, seq in enumerate(batch_strs):
            # HF tokenizer: <cls> en 0, secuencia, <eos> al final
            seq_len = len(seq)

            if output_hidden:
                # hidden_states es tupla (embedding + capa_1 ... capa_N)
                for layer_idx in repr_layers:
                    hidden = outputs.hidden_states[layer_idx][i, 1:seq_len+1].cpu().numpy()
                    if multi_layer:
                        all_residue_reps.setdefault(layer_idx, []).append(hidden)
                    else:
                        all_residue_reps.append(hidden)
            else:
                hidden = outputs.last_hidden_state[i, 1:seq_len+1].cpu().numpy()
                all_residue_reps.append(hidden)

            all_labels.append(batch_labels[i])

    return all_labels, all_residue_reps


def global_embedding_from_residue_matrix(residue_matrix):
    return residue_matrix.mean(axis=0)


def local_window_embedding(residue_matrix, pos_1based, k=5):
    idx = pos_1based - 1
    left = max(0, idx - k)
    right = min(residue_matrix.shape[0], idx + k + 1)
    return residue_matrix[left:right].mean(axis=0)


def residue_only_embedding(residue_matrix, pos_1based):
    idx = pos_1based - 1
    return residue_matrix[idx]


In [ ]:
all_sequences = [("WT", tp53_seq)] + [(m["mutant"], m["sequence"]) for m in mutants]

labels, residue_reps = get_residue_representations_batched(
    all_sequences,
    model=model,
    tokenizer=tokenizer,
    device=device,
    batch_size=64,
)

print("Number of sequences:", len(labels))
print("Residue matrix shape for WT:", residue_reps[0].shape)


In [ ]:
wt_emb = global_embedding_from_residue_matrix(residue_reps[0])
print("Global embedding matrix shape:", wt_emb.shape)

In [ ]:
global_emb = np.vstack([global_embedding_from_residue_matrix(x) for x in residue_reps])
print("Global embedding matrix shape:", global_emb.shape)

In [ ]:
k5_emb = np.vstack([wt_emb]+[local_window_embedding(x, m["position"], k=5) for x, m in zip(residue_reps, mutants)])
print("Local k=5 embedding matrix shape:", k5_emb.shape)

In [ ]:
wt_residue_reps = residue_reps[0]
wt_global_emb = global_embedding_from_residue_matrix(wt_residue_reps).reshape(1, -1)

results = []

for i, m in enumerate(mutants, start=1):
    mut_residue_reps = residue_reps[i]
    pos = m["position"]

    # Global
    mut_global_emb = global_embedding_from_residue_matrix(mut_residue_reps).reshape(1, -1)
    global_cos_sim = cosine_similarity(mut_global_emb, wt_global_emb)[0, 0]
    global_cos_dist = 1.0 - global_cos_sim
    global_euc_dist = euclidean_distances(mut_global_emb, wt_global_emb)[0, 0]

    # Local k=0
    wt_k0 = residue_only_embedding(wt_residue_reps, pos).reshape(1, -1)
    mut_k0 = residue_only_embedding(mut_residue_reps, pos).reshape(1, -1)
    k0_cos_sim = cosine_similarity(mut_k0, wt_k0)[0, 0]
    k0_cos_dist = 1.0 - k0_cos_sim
    k0_euc_dist = euclidean_distances(mut_k0, wt_k0)[0, 0]

    # Local k=3
    wt_k3 = local_window_embedding(wt_residue_reps, pos, k=3).reshape(1, -1)
    mut_k3 = local_window_embedding(mut_residue_reps, pos, k=3).reshape(1, -1)
    k3_cos_sim = cosine_similarity(mut_k3, wt_k3)[0, 0]
    k3_cos_dist = 1.0 - k3_cos_sim
    k3_euc_dist = euclidean_distances(mut_k3, wt_k3)[0, 0]

    # Local k=5
    wt_k5 = local_window_embedding(wt_residue_reps, pos, k=5).reshape(1, -1)
    mut_k5 = local_window_embedding(mut_residue_reps, pos, k=5).reshape(1, -1)
    k5_cos_sim = cosine_similarity(mut_k5, wt_k5)[0, 0]
    k5_cos_dist = 1.0 - k5_cos_sim
    k5_euc_dist = euclidean_distances(mut_k5, wt_k5)[0, 0]

    results.append({
        "mutant": m["mutant"],
        "position": m["position"],
        "wt_aa": m["wt_aa"],
        "mut_aa": m["mut_aa"],
        "group": m["group"],

        "global_cosine_similarity_to_WT": float(global_cos_sim),
        "global_cosine_distance_to_WT": float(global_cos_dist),
        "global_euclidean_distance_to_WT": float(global_euc_dist),

        "k0_cosine_similarity_to_WT": float(k0_cos_sim),
        "k0_cosine_distance_to_WT": float(k0_cos_dist),
        "k0_euclidean_distance_to_WT": float(k0_euc_dist),

        "k3_cosine_similarity_to_WT": float(k3_cos_sim),
        "k3_cosine_distance_to_WT": float(k3_cos_dist),
        "k3_euclidean_distance_to_WT": float(k3_euc_dist),

        "k5_cosine_similarity_to_WT": float(k5_cos_sim),
        "k5_cosine_distance_to_WT": float(k5_cos_dist),
        "k5_euclidean_distance_to_WT": float(k5_euc_dist),
    })

df = pd.DataFrame(results)
df.head()

In [ ]:
metric_cols = [
    "global_cosine_distance_to_WT",
    "k0_cosine_distance_to_WT",
    "k3_cosine_distance_to_WT",
    "k5_cosine_distance_to_WT",
]

df[metric_cols].describe().T

In [ ]:
df_metric_comparison = df[metric_cols].describe().T
df_metric_comparison['metric'] = df_metric_comparison.index
df_metric_comparison = df_metric_comparison.reset_index(drop=True)
df_metric_comparison = df_metric_comparison[['metric', 'mean', 'std', 'min', '50%', 'max']]
df_metric_comparison["metric"] = df_metric_comparison["metric"].str.replace("_", " ")
df_metric_comparison['mean'].round()
df_metric_comparison.to_csv("metric_comparison.csv", index=False,float_format='%.2f')
df_metric_comparison["metric"] = ["Global", "k0", "k3", "k5"]

df_metric_comparison

In [ ]:
df_metric_comparison.to_csv("metric_comparison.csv", index=False)

In [ ]:

plt.figure(figsize=(10, 6))
df_metric_comparison.set_index('metric')[['min', '50%', 'max']].T.boxplot()
plt.yscale('log')
plt.title('Distribución de Distancias por Métrica (Escala Logarítmica)')
plt.ylabel('Valor de la Distancia (log)')
plt.xticks(rotation=45, ha='right')
plt.grid(True, which="both", ls="--", alpha=0.5)
plt.tight_layout()
plt.savefig("metric_comparison_boxplot.svg")
plt.show()

In [ ]:
for col in metric_cols:
    print("=" * 80)
    print(col)
    display(
        df.sort_values(col, ascending=False)[
            ["mutant", "position", "group", col]
        ].head(10)
    )

In [ ]:
summary_by_position = (
    df.groupby(["group", "position", "wt_aa"], as_index=False)
      .agg(
          n_mutants=("mutant", "count"),

          global_mean=("global_cosine_distance_to_WT", "mean"),
          global_median=("global_cosine_distance_to_WT", "median"),
          global_std=("global_cosine_distance_to_WT", "std"),
          global_max=("global_cosine_distance_to_WT", "max"),

          k0_mean=("k0_cosine_distance_to_WT", "mean"),
          k0_median=("k0_cosine_distance_to_WT", "median"),
          k0_std=("k0_cosine_distance_to_WT", "std"),
          k0_max=("k0_cosine_distance_to_WT", "max"),

          k3_mean=("k3_cosine_distance_to_WT", "mean"),
          k3_median=("k3_cosine_distance_to_WT", "median"),
          k3_std=("k3_cosine_distance_to_WT", "std"),
          k3_max=("k3_cosine_distance_to_WT", "max"),

          k5_mean=("k5_cosine_distance_to_WT", "mean"),
          k5_median=("k5_cosine_distance_to_WT", "median"),
          k5_std=("k5_cosine_distance_to_WT", "std"),
          k5_max=("k5_cosine_distance_to_WT", "max"),
      )
)

summary_by_position["site"] = summary_by_position["wt_aa"] + summary_by_position["position"].astype(str)
summary_by_position.sort_values("k5_mean", ascending=False)

In [ ]:
mutants_df = pd.DataFrame(mutants)
labels_df = pd.DataFrame(labels, columns=["label"])
group_list = [m["group"] for m in mutants]
group_list.append("WT")


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import umap

from sklearn.preprocessing import StandardScaler


# =========================
# 1. Revisar dimensiones
# =========================

X = global_emb
labels = group_list

print("Shape embeddings:", X.shape)
print("Número de labels:", len(labels))

assert X.shape[0] == len(labels), "El número de embeddings y labels no coincide"


# =========================
# 2. Escalar embeddings
# =========================

X_scaled = StandardScaler().fit_transform(X)


# =========================
# 3. Calcular UMAP
# =========================

reducer = umap.UMAP(
    n_components=2,
    n_neighbors=15,
    min_dist=0.1,
    metric="cosine",
)

X_umap = reducer.fit_transform(X_scaled)


# =========================
# 4. Pasar a dataframe
# =========================

df_umap = pd.DataFrame({
    "UMAP1": X_umap[:, 0],
    "UMAP2": X_umap[:, 1],
    "label": labels,
})


# =========================
# 5. Graficar por label
# =========================

plt.figure(figsize=(7, 6))

for label in np.unique(labels):
    subset = df_umap[df_umap["label"] == label]
    plt.scatter(
        subset["UMAP1"],
        subset["UMAP2"],
        label=label,
        alpha=0.8,
        s=45,
    )

plt.xlabel("UMAP 1")
plt.ylabel("UMAP 2")
plt.title("UMAP de embeddings locales ESM-2")
plt.legend(title="Sitio")
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import umap


X_scaled = StandardScaler().fit_transform(X)

X_pca = PCA(n_components=50, random_state=42).fit_transform(X_scaled)

X_umap = umap.UMAP(
    n_components=2,
    n_neighbors=15,
    min_dist=0.1,
    metric="cosine",
    random_state=42,
).fit_transform(X_pca)

# =========================
# 4. Pasar a dataframe
# =========================

df_umap = pd.DataFrame({
    "UMAP1": X_umap[:, 0],
    "UMAP2": X_umap[:, 1],
    "label": labels,
})


# =========================
# 5. Graficar por label
# =========================

plt.figure(figsize=(7, 6))

for label in np.unique(labels):
    subset = df_umap[df_umap["label"] == label]
    plt.scatter(
        subset["UMAP1"],
        subset["UMAP2"],
        label=label,
        alpha=0.8,
        s=45,
    )

plt.xlabel("UMAP 1")
plt.ylabel("UMAP 2")
plt.title("PCA + UMAP de embeddings ESM-2")
plt.legend(title="Label")
plt.tight_layout()
plt.show()


In [ ]:

from sklearn.manifold import TSNE


coords = TSNE(n_components=2).fit_transform(global_emb)

plot_df = pd.DataFrame({
    "label": labels,
    "PC1": coords[:, 0],
    "PC2": coords[:, 1],
    "group": ["WT"] + [m["group"] for m in mutants]
})

plt.figure(figsize=(8, 6))
for g in np.unique(labels):
    sub = plot_df[plot_df["group"] == g]
    if len(sub) > 0:
        plt.scatter(sub["PC1"], sub["PC2"], label=g, alpha=0.75)

wt_row = plot_df.iloc[0]
plt.text(wt_row["PC1"], wt_row["PC2"], "WT", fontsize=10)

plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("TSNE of global embeddings")
plt.legend()
plt.savefig("tsne-global-embeddings.svg")
plt.show()

In [ ]:
pca = PCA(n_components=2)
coords = pca.fit_transform(global_emb)

plot_df = pd.DataFrame({
    "label": labels,
    "PC1": coords[:, 0],
    "PC2": coords[:, 1],
    "group": ["WT"] + [m["group"] for m in mutants]
})

plt.figure(figsize=(8, 6))
for g in np.unique(labels):
    sub = plot_df[plot_df["group"] == g]
    if len(sub) > 0:
        plt.scatter(sub["PC1"], sub["PC2"], label=g, alpha=0.75)

wt_row = plot_df.iloc[0]
plt.text(wt_row["PC1"], wt_row["PC2"], "WT", fontsize=10)

plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("PCA of global embeddings")
plt.legend()
plt.savefig("pca-global-embeddings.svg")
plt.show()

In [ ]:
## Proyeccion de k5

pca = PCA(n_components=2)
coords = pca.fit_transform(k5_emb)

plot_df = pd.DataFrame({
    "label": labels,
    "PC1": coords[:, 0],
    "PC2": coords[:, 1],
    "group": ["WT"] + [m["group"] for m in mutants]
})

plt.figure(figsize=(8, 6))
for g in np.unique(labels):
    sub = plot_df[plot_df["group"] == g]
    if len(sub) > 0:
        plt.scatter(sub["PC1"], sub["PC2"], label=g, alpha=0.75)

wt_row = plot_df.iloc[0]
plt.text(wt_row["PC1"], wt_row["PC2"], "WT", fontsize=10)

plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("PCA of k5 embeddings")
plt.legend()
plt.savefig("pca-k5-embeddings.svg")
plt.show()

In [ ]:
coords = TSNE(n_components=2,metric='cosine').fit_transform(k5_emb)

plot_df = pd.DataFrame({
    "label": labels,
    "PC1": coords[:, 0],
    "PC2": coords[:, 1],
    "group": ["WT"] + [m["group"] for m in mutants]
})

plt.figure(figsize=(8, 6))
for g in np.unique(labels):
    sub = plot_df[plot_df["group"] == g]
    if len(sub) > 0:
        plt.scatter(sub["PC1"], sub["PC2"], label=g, alpha=0.75)

wt_row = plot_df.iloc[0]
plt.text(wt_row["PC1"], wt_row["PC2"], "WT", fontsize=10)

plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("TSNE of k5 embeddings")
plt.legend()
plt.savefig("tsne-k5-embeddings.svg")
plt.show()

In [ ]:

# =========================
# 1. Revisar dimensiones
# =========================

X = k5_emb
labels = group_list

print("Shape embeddings:", X.shape)
print("Número de labels:", len(labels))

assert X.shape[0] == len(labels), "El número de embeddings y labels no coincide"


# =========================
# 2. Escalar embeddings
# =========================

X_scaled = StandardScaler().fit_transform(X)


# =========================
# 3. Calcular UMAP
# =========================

reducer = umap.UMAP(
    n_components=2,
    n_neighbors=15,
    min_dist=0.1,
    metric="cosine",
)

X_umap = reducer.fit_transform(X_scaled)


# =========================
# 4. Pasar a dataframe
# =========================

df_umap = pd.DataFrame({
    "UMAP1": X_umap[:, 0],
    "UMAP2": X_umap[:, 1],
    "label": labels,
})


# =========================
# 5. Graficar por label
# =========================


In [ ]:

plt.figure(figsize=(7, 6))

for label in np.unique(labels):
    subset = df_umap[df_umap["label"] == label]
    if label == "WT":
        plt.scatter(
            subset["UMAP1"],
            subset["UMAP2"],
            label=label,
            alpha=1.0,
            s=200,
            marker="x"
        )
    else:
      plt.scatter(
          subset["UMAP1"],
          subset["UMAP2"],
          label=label,
          alpha=0.8,
          s=45,
      )

plt.xlabel("UMAP 1")
plt.ylabel("UMAP 2")
plt.title("UMAP de embeddings ESM-2")
plt.legend(title="Label")
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 4), sharey=False)

plot_map = [
    ("global_cosine_distance_to_WT", "Global"),
    ("k0_cosine_distance_to_WT", "Local k=0"),
    ("k3_cosine_distance_to_WT", "Local k=3"),
    ("k5_cosine_distance_to_WT", "Local k=5"),
]

for ax, (col, title) in zip(axes, plot_map):
    df.boxplot(column=col, by="group", grid=False, ax=ax)
    ax.set_title(title)
    ax.set_xlabel("")
    ax.set_ylabel("Cosine distance")
    ax.tick_params(axis="x", rotation=15)

plt.suptitle("")
plt.tight_layout()
plt.show()

In [ ]:
rank_global = summary_by_position.sort_values("global_mean", ascending=False)[["site", "global_mean"]].reset_index(drop=True)
rank_k5 = summary_by_position.sort_values("k5_mean", ascending=False)[["site", "k5_mean"]].reset_index(drop=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

tmp = summary_by_position.sort_values("global_mean", ascending=True)
axes[0].barh(tmp["site"], tmp["global_mean"])
axes[0].set_title("Average sensitivity by position (Global)")
axes[0].set_xlabel("Mean cosine distance to WT")
axes[0].set_ylabel("Position")

tmp = summary_by_position.sort_values("k5_mean", ascending=True)
axes[1].barh(tmp["site"], tmp["k5_mean"])
axes[1].set_title("Average sensitivity by position (Local k=5)")
axes[1].set_xlabel("Mean cosine distance to WT")
axes[1].set_ylabel("Position")

plt.tight_layout()
plt.savefig("average-sensitivity-by-position-global-vs-k5.svg")
plt.show()

In [ ]:
compare_rank = summary_by_position[["site", "group", "global_mean", "k5_mean"]].copy()
compare_rank["delta_k5_minus_global"] = compare_rank["k5_mean"] - compare_rank["global_mean"]
compare_rank = compare_rank.sort_values("delta_k5_minus_global", ascending=False)

compare_rank

In [ ]:
compare_rank.to_csv("compare_rank.csv", index=False)



In [ ]:
tmp = compare_rank.sort_values("delta_k5_minus_global", ascending=True)

plt.figure(figsize=(8, 6))
plt.barh(tmp["site"], tmp["delta_k5_minus_global"])
plt.xlabel("k5_promedio- promedio-global ")
plt.ylabel("Posicion")
plt.title("Diferencia de sensibilidad entre los encajes globales y locales(k=5)")
plt.savefig("compare_rank.svg")
plt.show()

In [ ]:
def plot_heatmap(df, value_col, title):
    heat = df.pivot(index="mut_aa", columns="position", values=value_col)

    plt.figure(figsize=(10, 5))
    plt.imshow(heat.values, aspect="auto")
    plt.xticks(range(len(heat.columns)), heat.columns)
    plt.yticks(range(len(heat.index)), heat.index)
    plt.colorbar(label=value_col)
    plt.xlabel("Position")
    plt.ylabel("Mutant amino acid")
    plt.title(title)
    plt.show()

plot_heatmap(df, "global_cosine_distance_to_WT", "Global cosine distance heatmap")
plot_heatmap(df, "k0_cosine_distance_to_WT", "Local k=0 cosine distance heatmap")
plot_heatmap(df, "k3_cosine_distance_to_WT", "Local k=3 cosine distance heatmap")
plot_heatmap(df, "k5_cosine_distance_to_WT", "Local k=5 cosine distance heatmap")

In [ ]:
df.sort_values("k5_cosine_distance_to_WT", ascending=False)[
    [
        "mutant", "position", "group",
        "global_cosine_distance_to_WT",
        "k0_cosine_distance_to_WT",
        "k3_cosine_distance_to_WT",
        "k5_cosine_distance_to_WT",
    ]
].head(25)

In [ ]:
df.to_csv("tp53_synthetic_mutants_embedding_metrics.csv", index=False)
summary_by_position.to_csv("tp53_position_summary_embedding_metrics.csv", index=False)

print("Saved:")
print("- tp53_synthetic_mutants_embedding_metrics.csv")
print("- tp53_position_summary_embedding_metrics.csv")

## Revision de cargas

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Carga neta simplificada
aa_charge = {
    "R": "positive", "K": "positive", "H": "positive",
    "D": "negative", "E": "negative",
    "A": "neutral", "C": "neutral", "F": "neutral", "G": "neutral",
    "I": "neutral", "L": "neutral", "M": "neutral", "N": "neutral",
    "P": "neutral", "Q": "neutral", "S": "neutral", "T": "neutral",
    "V": "neutral", "W": "neutral", "Y": "neutral"
}

# Clase fisicoquímica gruesa
aa_class = {
    "A": "hydrophobic", "V": "hydrophobic", "I": "hydrophobic",
    "L": "hydrophobic", "M": "hydrophobic", "F": "hydrophobic",
    "W": "hydrophobic", "Y": "hydrophobic",

    "S": "polar", "T": "polar", "N": "polar", "Q": "polar",
    "C": "polar", "G": "special", "P": "special",

    "D": "charged", "E": "charged", "K": "charged",
    "R": "charged", "H": "charged"
}

# Tamaño aproximado
aa_size = {
    "G": "small", "A": "small", "S": "small", "C": "small",
    "D": "medium", "P": "medium", "N": "medium", "T": "medium",
    "E": "medium", "V": "medium", "Q": "medium", "H": "medium",
    "I": "large", "L": "large", "M": "large", "K": "large",
    "R": "large", "F": "large", "Y": "large", "W": "large"
}

aromatic = {"F", "W", "Y"}
aliphatic = {"A", "V", "I", "L", "M"}
polar = {"S", "T", "N", "Q", "C"}
charged = {"D", "E", "K", "R", "H"}

def annotate_mutation_type(df):
    df = df.copy()

    # Propiedades WT y MUT
    df["wt_charge"] = df["wt_aa"].map(aa_charge)
    df["mut_charge"] = df["mut_aa"].map(aa_charge)

    df["wt_class"] = df["wt_aa"].map(aa_class)
    df["mut_class"] = df["mut_aa"].map(aa_class)

    df["wt_size"] = df["wt_aa"].map(aa_size)
    df["mut_size"] = df["mut_aa"].map(aa_size)

    # Cambios binarios
    df["charge_change"] = df["wt_charge"] != df["mut_charge"]
    df["class_change"] = df["wt_class"] != df["mut_class"]
    df["size_change"] = df["wt_size"] != df["mut_size"]

    # Casos especiales
    df["introduces_proline"] = df["mut_aa"] == "P"
    df["introduces_glycine"] = df["mut_aa"] == "G"
    df["introduces_cysteine"] = df["mut_aa"] == "C"

    df["removes_proline"] = df["wt_aa"] == "P"
    df["removes_glycine"] = df["wt_aa"] == "G"
    df["removes_cysteine"] = df["wt_aa"] == "C"

    # Aromaticidad
    df["wt_aromatic"] = df["wt_aa"].isin(aromatic)
    df["mut_aromatic"] = df["mut_aa"].isin(aromatic)
    df["aromatic_change"] = df["wt_aromatic"] != df["mut_aromatic"]

    # Hidrofobicidad gruesa
    def hydro_group(aa):
        if aa in aliphatic or aa in aromatic:
            return "hydrophobic"
        elif aa in polar:
            return "polar"
        elif aa in charged:
            return "charged"
        elif aa in {"G", "P"}:
            return "special"
        else:
            return "other"

    df["wt_hydro_group"] = df["wt_aa"].map(hydro_group)
    df["mut_hydro_group"] = df["mut_aa"].map(hydro_group)
    df["hydro_group_change"] = df["wt_hydro_group"] != df["mut_hydro_group"]

    # Etiqueta resumida de conservatividad gruesa
    df["is_conservative_like"] = (
        (~df["charge_change"]) &
        (~df["class_change"]) &
        (~df["size_change"])
    )

    return df

df_annot = annotate_mutation_type(df)
df_annot.head()

In [ ]:
summary_flags = {
    "charge_change": df_annot["charge_change"].value_counts(),
    "class_change": df_annot["class_change"].value_counts(),
    "size_change": df_annot["size_change"].value_counts(),
    "hydro_group_change": df_annot["hydro_group_change"].value_counts(),
    "introduces_proline": df_annot["introduces_proline"].value_counts(),
    "introduces_glycine": df_annot["introduces_glycine"].value_counts(),
    "introduces_cysteine": df_annot["introduces_cysteine"].value_counts(),
    "aromatic_change": df_annot["aromatic_change"].value_counts(),
    "is_conservative_like": df_annot["is_conservative_like"].value_counts(),
}

for k, v in summary_flags.items():
    print("=" * 60)
    print(k)
    print(v)

In [ ]:
metric = "k5_cosine_distance_to_WT"

fig, axes = plt.subplots(2, 3, figsize=(16, 9))

plot_specs = [
    ("charge_change", "Charge change"),
    ("class_change", "Class change"),
    ("size_change", "Size change"),
    ("hydro_group_change", "Hydrophobicity group change"),
    ("introduces_proline", "Introduces proline"),
    ("introduces_glycine", "Introduces glycine"),
]

for ax, (col, title) in zip(axes.flatten(), plot_specs):
    df_annot.boxplot(column=metric, by=col, grid=False, ax=ax)
    ax.set_title(title)
    ax.set_xlabel("")
    ax.set_ylabel(metric)
    ax.tick_params(axis="x", rotation=0)

plt.suptitle("")
plt.tight_layout()
plt.savefig("boxplot-categories-vs-k5-cosine-distance.svg")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

extra_specs = [
    ("introduces_cysteine", "Introduces cysteine"),
    ("aromatic_change", "Aromatic change"),
    ("is_conservative_like", "Conservative-like substitution"),
]

for ax, (col, title) in zip(axes.flatten(), extra_specs):
    df_annot.boxplot(column=metric, by=col, grid=False, ax=ax)
    ax.set_title(title)
    ax.set_xlabel("")
    ax.set_ylabel(metric)

plt.suptitle("")
plt.tight_layout()
plt.savefig("boxplot-categories-vs-k5-cosine-distance2.svg")
plt.show()

In [ ]:
def summarize_by_flag(df, flag_col, metric_col="k5_cosine_distance_to_WT"):
    out = (
        df.groupby(flag_col)[metric_col]
          .agg(["count", "mean", "median", "std", "max"])
          .reset_index()
          .sort_values("mean", ascending=False)
    )
    out["flag"] = flag_col
    return out

summary_tables = []
for col in [
    "charge_change",
    "class_change",
    "size_change",
    "hydro_group_change",
    "introduces_proline",
    "introduces_glycine",
    "introduces_cysteine",
    "aromatic_change",
    "is_conservative_like",
]:
    summary_tables.append(summarize_by_flag(df_annot, col, metric))

summary_flags_df = pd.concat(summary_tables, ignore_index=True)
summary_flags_df

In [ ]:
cols_to_show = [
    "mutant", "position", "group",
    "wt_aa", "mut_aa",
    "k5_cosine_distance_to_WT",
    "charge_change", "class_change", "size_change",
    "introduces_proline", "introduces_glycine", "introduces_cysteine",
    "aromatic_change", "is_conservative_like"
]

print("Top mutantes que introducen prolina")
display(
    df_annot[df_annot["introduces_proline"]]
    .sort_values("k5_cosine_distance_to_WT", ascending=False)[cols_to_show]
    .head(15)
)

print("Top mutantes que introducen glicina")
display(
    df_annot[df_annot["introduces_glycine"]]
    .sort_values("k5_cosine_distance_to_WT", ascending=False)[cols_to_show]
    .head(15)
)

print("Top mutantes conservativas-like")
display(
    df_annot[df_annot["is_conservative_like"]]
    .sort_values("k5_cosine_distance_to_WT", ascending=False)[cols_to_show]
    .head(15)
)

In [ ]:
df_annot.to_csv("tp53_synthetic_mutants_embedding_metrics_annotated.csv", index=False)
summary_flags_df.to_csv("tp53_mutation_type_summary_k5.csv", index=False)

print("Saved:")
print("- tp53_synthetic_mutants_embedding_metrics_annotated.csv")
print("- tp53_mutation_type_summary_k5.csv")

## Pruebas estadisticas


In [ ]:
!pip -q install scipy statsmodels

In [ ]:
import numpy as np
import pandas as pd

from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests

In [ ]:
metric = "k5_cosine_distance_to_WT"

flag_cols = [
    "charge_change",
    "class_change",
    "size_change",
    "hydro_group_change",
    "introduces_proline",
    "introduces_glycine",
    "introduces_cysteine",
    "aromatic_change",
    "is_conservative_like",
]

df_annot[flag_cols + [metric]].head()

In [ ]:
def mann_whitney_for_flag(df, flag_col, metric_col):
    sub = df[[flag_col, metric_col]].dropna().copy()

    x_false = sub.loc[sub[flag_col] == False, metric_col].values
    x_true  = sub.loc[sub[flag_col] == True,  metric_col].values

    n_false = len(x_false)
    n_true = len(x_true)

    if n_false == 0 or n_true == 0:
        return {
            "flag": flag_col,
            "n_false": n_false,
            "n_true": n_true,
            "mean_false": np.nan,
            "mean_true": np.nan,
            "median_false": np.nan,
            "median_true": np.nan,
            "median_diff_true_minus_false": np.nan,
            "u_stat": np.nan,
            "p_value": np.nan,
            "rank_biserial": np.nan,
        }

    # Mann–Whitney U, dos colas
    u_stat, p_value = mannwhitneyu(x_true, x_false, alternative="two-sided")

    # Rank-biserial correlation
    # RBC = 2U/(n1*n2) - 1, usando U del grupo "true"
    rank_biserial = (2 * u_stat) / (n_true * n_false) - 1

    return {
        "flag": flag_col,
        "n_false": n_false,
        "n_true": n_true,
        "mean_false": float(np.mean(x_false)),
        "mean_true": float(np.mean(x_true)),
        "median_false": float(np.median(x_false)),
        "median_true": float(np.median(x_true)),
        "median_diff_true_minus_false": float(np.median(x_true) - np.median(x_false)),
        "u_stat": float(u_stat),
        "p_value": float(p_value),
        "rank_biserial": float(rank_biserial),
    }

In [ ]:
mw_results = pd.DataFrame(
    [mann_whitney_for_flag(df_annot, col, metric) for col in flag_cols]
)

# Corrección Benjamini–Hochberg
valid_mask = mw_results["p_value"].notna()
rej, p_adj, _, _ = multipletests(
    mw_results.loc[valid_mask, "p_value"].values,
    alpha=0.05,
    method="fdr_bh"
)

mw_results.loc[valid_mask, "p_adj_bh"] = p_adj
mw_results.loc[valid_mask, "significant_bh_0_05"] = rej

mw_results = mw_results.sort_values("p_adj_bh", ascending=True)
mw_results

In [ ]:
mw_display = mw_results.copy()

for col in [
    "mean_false", "mean_true",
    "median_false", "median_true",
    "median_diff_true_minus_false",
    "rank_biserial",
    "p_value", "p_adj_bh"
]:
    mw_display[col] = mw_display[col].round(6)

mw_display

In [ ]:
def interpret_rbc(rbc):
    a = abs(rbc)
    if a < 0.1:
        return "very small"
    elif a < 0.3:
        return "small"
    elif a < 0.5:
        return "moderate"
    else:
        return "large"

mw_interpret = mw_results.copy()
mw_interpret["effect_size_label"] = mw_interpret["rank_biserial"].apply(
    lambda x: interpret_rbc(x) if pd.notna(x) else np.nan
)

mw_interpret[[
    "flag",
    "median_false",
    "median_true",
    "median_diff_true_minus_false",
    "rank_biserial",
    "effect_size_label",
    "p_value",
    "p_adj_bh",
    "significant_bh_0_05",
]].sort_values("p_adj_bh")

In [ ]:

mw_interpret.to_csv("mannwhitney_k5_results.csv", index=False, encoding='utf-8')
print("Saved: mannwhitney_k5_results.csv")

In [ ]:
import matplotlib.pyplot as plt

tmp = mw_results.sort_values("rank_biserial", ascending=True)

plt.figure(figsize=(8, 5))
plt.barh(tmp["flag"], tmp["rank_biserial"])
plt.axvline(0, linestyle="--")
plt.xlabel("Correlacion rango biserial")
plt.ylabel("Categoria")
plt.title(f"Efectos de la prueba de MW para los embeddings k5")
plt.savefig("mannwhitney-effect-sizes-k5-cosine.svg")
plt.show()

In [ ]:
tmp = mw_results.copy()
tmp["median_true_minus_false"] = tmp["median_true"] - tmp["median_false"]
tmp = tmp.sort_values("median_true_minus_false", ascending=True)

plt.figure(figsize=(8, 5))
plt.barh(tmp["flag"], tmp["median_true_minus_false"])
plt.axvline(0, linestyle="--")
plt.xlabel("Median(True) - Median(False)")
plt.ylabel("Flag")
plt.title(f"Median shift in {metric}")
plt.savefig("median-shift-k5-cosine.svg")
plt.show()

In [ ]:
mw_results.to_csv("tp53_mannwhitney_k5_results.csv", index=False)
print("Saved: tp53_mannwhitney_k5_results.csv")

#FASE 2


In [ ]:
!pip -q install requests scipy statsmodels

In [ ]:
import io
import re
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import spearmanr, pearsonr, mannwhitneyu

In [ ]:
# Score set recomendado para empezar
MAVEDB_URN = "urn:mavedb:00000068-0-1"

base = "https://api.mavedb.org/api/v1"
url_scores = f"{base}/score-sets/{MAVEDB_URN}/scores"
url_meta = f"{base}/score-sets/{MAVEDB_URN}"

meta = requests.get(url_meta, timeout=60)
meta.raise_for_status()
meta_json = meta.json()

scores = requests.get(url_scores, timeout=60)
scores.raise_for_status()

mave_raw = pd.read_csv(io.StringIO(scores.text))

print("Title:", meta_json.get("title", "N/A"))
print("URN:", MAVEDB_URN)
print("Shape:", mave_raw.shape)
print("\nColumns:")
print(list(mave_raw.columns))
display(mave_raw.head())

In [ ]:
def pick_first_existing(columns, candidates):
    for c in candidates:
        if c in columns:
            return c
    return None

variant_candidates = [
    "hgvs_pro", "hgvs_p", "protein_variant", "variant",
    "aa_variant", "amino_acid_change", "protein_change"
]

score_candidates = [
    "score", "main_score", "functional_score",
    "score_value", "normalized_score"
]

variant_col = pick_first_existing(mave_raw.columns, variant_candidates)
score_col = pick_first_existing(mave_raw.columns, score_candidates)

print("Auto-detected variant_col:", variant_col)
print("Auto-detected score_col:", score_col)

if variant_col is None:
    print("\nNo pude detectar automáticamente la columna de variante.")
    print("Columnas disponibles:")
    print(list(mave_raw.columns))

if score_col is None:
    print("\nNo pude detectar automáticamente la columna de score.")
    numeric_cols = [c for c in mave_raw.columns if pd.api.types.is_numeric_dtype(mave_raw[c])]
    print("Columnas numéricas candidatas:")
    print(numeric_cols)

In [ ]:
aa3_to_aa1 = {
    "Ala": "A", "Arg": "R", "Asn": "N", "Asp": "D", "Cys": "C",
    "Gln": "Q", "Glu": "E", "Gly": "G", "His": "H", "Ile": "I",
    "Leu": "L", "Lys": "K", "Met": "M", "Phe": "F", "Pro": "P",
    "Ser": "S", "Thr": "T", "Trp": "W", "Tyr": "Y", "Val": "V",
    "Ter": "*", "Stop": "*"
}

def normalize_protein_variant(x):
    if pd.isna(x):
        return None

    s = str(x).strip()

    # quitar prefijo p.
    s = re.sub(r"^p\.", "", s)

    # Caso 1: una letra, p. R175H
    m1 = re.match(r"^([A-Z\*])(\d+)([A-Z\*])$", s)
    if m1:
        wt, pos, mut = m1.groups()
        return f"{wt}{pos}{mut}"

    # Caso 2: tres letras, p.Arg175His
    m2 = re.match(r"^([A-Z][a-z]{2})(\d+)([A-Z][a-z]{2}|Ter|Stop)$", s)
    if m2:
        wt3, pos, mut3 = m2.groups()
        wt = aa3_to_aa1.get(wt3)
        mut = aa3_to_aa1.get(mut3)
        if wt is not None and mut is not None:
            return f"{wt}{pos}{mut}"

    return None

mave = mave_raw.copy()
mave["mutant"] = mave[variant_col].apply(normalize_protein_variant)
mave["score_raw"] = pd.to_numeric(mave[score_col], errors="coerce")

display(mave[[variant_col, "mutant", score_col, "score_raw"]].head(20))
print("Mutantes parseados:", mave["mutant"].notna().sum())

In [ ]:
missense_pattern = re.compile(r"^[ACDEFGHIKLMNPQRSTVWY](\d+)[ACDEFGHIKLMNPQRSTVWY]$")

mave_missense = mave[
    mave["mutant"].notna() &
    mave["mutant"].str.match(missense_pattern) &
    mave["score_raw"].notna()
].copy()

# Si hay duplicados por mutante, promediamos
mave_missense = (
    mave_missense.groupby("mutant", as_index=False)
    .agg(score_raw=("score_raw", "mean"))
)

print("MAVE missense shape:", mave_missense.shape)
display(mave_missense.head())

In [ ]:
merge_cols = [
    "mutant", "position", "wt_aa", "mut_aa", "group",
    "global_cosine_distance_to_WT",
    "k0_cosine_distance_to_WT",
    "k3_cosine_distance_to_WT",
    "k5_cosine_distance_to_WT",
    "class_change", "hydro_group_change", "is_conservative_like",
    "introduces_glycine", "introduces_proline", "introduces_cysteine"
]

tp53_panel_real = df_annot[merge_cols].merge(
    mave_missense,
    on="mutant",
    how="inner"
)

print("Overlap con panel:", tp53_panel_real.shape)
display(tp53_panel_real.head())
print("\nNúmero de posiciones con datos reales:")
print(tp53_panel_real["position"].value_counts().sort_index())

In [ ]:
canonical_bad = ["R175H", "G245S", "R248Q", "R248W", "R273H", "R282W"]

check = tp53_panel_real[tp53_panel_real["mutant"].isin(canonical_bad)].copy()
display(check[["mutant", "score_raw", "k5_cosine_distance_to_WT"]].sort_values("mutant"))

print(
    "Interpreta esta tabla así:\n"
    "- si estos mutantes dañinos salen con scores bajos, entonces score_raw probablemente ya significa 'más funcional = mayor'.\n"
    "- si salen con scores altos, convendrá invertir el signo."
)

In [ ]:
flip_score = False  # cambia a True si ves que el score está invertido

tp53_panel_real = tp53_panel_real.copy()
tp53_panel_real["functional_score"] = (
    -tp53_panel_real["score_raw"] if flip_score else tp53_panel_real["score_raw"]
)

display(
    tp53_panel_real[["mutant", "score_raw", "functional_score"]].head()
)

In [ ]:
def corr_report(x, y, label_x):
    mask = np.isfinite(x) & np.isfinite(y)
    x = np.asarray(x)[mask]
    y = np.asarray(y)[mask]

    sp = spearmanr(x, y)
    pr = pearsonr(x, y)

    return {
        "metric": label_x,
        "n": len(x),
        "spearman_rho": sp.statistic,
        "spearman_p": sp.pvalue,
        "pearson_r": pr.statistic,
        "pearson_p": pr.pvalue,
    }

corr_table = pd.DataFrame([
    corr_report(tp53_panel_real["global_cosine_distance_to_WT"], tp53_panel_real["functional_score"], "global"),
    corr_report(tp53_panel_real["k3_cosine_distance_to_WT"], tp53_panel_real["functional_score"], "k3"),
    corr_report(tp53_panel_real["k5_cosine_distance_to_WT"], tp53_panel_real["functional_score"], "k5"),
])

corr_table



In [ ]:
corr_table.to_csv("correlation-embedding-table.csv")

In [ ]:
x = tp53_panel_real["k5_cosine_distance_to_WT"].values
y = tp53_panel_real["functional_score"].values

rho, pval = spearmanr(x, y)

plt.figure(figsize=(7, 5))
plt.scatter(x, y, alpha=0.8)
plt.xlabel("Local embedding shift (k=5)")
plt.ylabel("Functional score")
plt.title(f"TP53: k=5 vs functional score\nSpearman rho={rho:.3f}, p={pval:.3g} modelo ")
plt.savefig("embedding-vs-functional-score-k5.svg")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

x = tp53_panel_real["global_cosine_distance_to_WT"].values
y = tp53_panel_real["functional_score"].values
rho, pval = spearmanr(x, y)
axes[0].scatter(x, y, alpha=0.8)
axes[0].set_xlabel("Global embedding shift")
axes[0].set_ylabel("Functional score")
axes[0].set_title(f"Global\nrho={rho:.3f}, p={pval:.3g}")

x = tp53_panel_real["k5_cosine_distance_to_WT"].values
y = tp53_panel_real["functional_score"].values
rho, pval = spearmanr(x, y)
axes[1].scatter(x, y, alpha=0.8)
axes[1].set_xlabel("Local embedding shift (k=5)")
axes[1].set_ylabel("Functional score")
axes[1].set_title(f"Local k=5\nrho={rho:.3f}, p={pval:.3g}")

plt.tight_layout()
plt.savefig("embedding-vs-functional-score-global-k5.svg")
plt.show()

In [ ]:
# Actualizamos las etiquetas antes de graficar el histograma/boxplot por terciles funcionales
tp53_panel_real = tp53_panel_real.copy()

# Aseguramos que los nombres de los grupos en el dataframe coincidan con los nuevos nombres
group_rename_map = {
    "hotspot": "DNA Binding Hotspots",
    "structural": "Structural Stability Sites",
    "comparison_dbd": "Other DBD Surface Sites"
}
tp53_panel_real["group"] = tp53_panel_real["group"].replace(group_rename_map)

# Creación de terciles funcionales (Low, Mid, High activity)
tp53_panel_real["functional_bin"] = pd.qcut(
    tp53_panel_real["functional_score"],
    q=3,
    labels=["low", "mid", "high"]
)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

tp53_panel_real.boxplot(column="global_cosine_distance_to_WT", by="functional_bin", grid=False, ax=axes[0])
axes[0].set_title("Global shift by functional bin")
axes[0].set_xlabel("Activity Level")
axes[0].set_ylabel("Embedding shift")

tp53_panel_real.boxplot(column="k5_cosine_distance_to_WT", by="functional_bin", grid=False, ax=axes[1])
axes[1].set_title("Local k=5 shift by functional bin")
axes[1].set_xlabel("Activity Level")
axes[1].set_ylabel("Embedding shift")

plt.suptitle("Distribution of Embedding Shifts by Functional Status (MAVEDB)")
plt.tight_layout()
plt.savefig("histogram-embeding-shift.svg")
plt.show()

In [ ]:
low = tp53_panel_real.loc[tp53_panel_real["functional_bin"] == "low", "k5_cosine_distance_to_WT"].values
high = tp53_panel_real.loc[tp53_panel_real["functional_bin"] == "high", "k5_cosine_distance_to_WT"].values

u, p = mannwhitneyu(low, high, alternative="two-sided")

print("Mann–Whitney low vs high on k5")
print("n_low =", len(low))
print("n_high =", len(high))
print("U =", u)
print("p =", p)
print("median_low =", np.median(low))
print("median_high =", np.median(high))

In [ ]:
tp53_panel_real["abs_k5"] = tp53_panel_real["k5_cosine_distance_to_WT"].abs()
tp53_panel_real["abs_score"] = tp53_panel_real["functional_score"].abs()

display(
    tp53_panel_real.sort_values("k5_cosine_distance_to_WT", ascending=False)[
        ["mutant", "position", "group", "k5_cosine_distance_to_WT", "functional_score"]
    ].head(20)
)

display(
    tp53_panel_real.sort_values("functional_score", ascending=True)[
        ["mutant", "position", "group", "k5_cosine_distance_to_WT", "functional_score"]
    ].head(20)
)

In [ ]:
corr_table.to_csv("tp53_panel_functional_correlations.csv", index=False)
tp53_panel_real.to_csv("tp53_panel_with_functional_scores.csv", index=False)

print("Saved:")
print("- tp53_panel_functional_correlations.csv")
print("- tp53_panel_with_functional_scores.csv")

In [ ]:
## Correlaciones

from scipy.stats import spearmanr, pearsonr, kendalltau
import pandas as pd
import numpy as np

In [ ]:
def correlation_report(df, x_col, y_col, label=None):
    sub = df[[x_col, y_col]].dropna().copy()

    x = sub[x_col].values
    y = sub[y_col].values

    sp = spearmanr(x, y)
    pr = pearsonr(x, y)
    kt = kendalltau(x, y)

    return {
        "label": label if label is not None else x_col,
        "n": len(sub),
        "spearman_rho": sp.statistic,
        "spearman_p": sp.pvalue,
        "pearson_r": pr.statistic,
        "pearson_p": pr.pvalue,
        "kendall_tau": kt.statistic,
        "kendall_p": kt.pvalue,
    }

In [ ]:
score_col = "score_raw"   # o "combined_lof_dn_score" / "activity_score"

corr_table = pd.DataFrame([
    correlation_report(tp53_panel_real, "global_cosine_distance_to_WT", score_col, "global"),
    correlation_report(tp53_panel_real, "k3_cosine_distance_to_WT", score_col, "local_k3"),
    correlation_report(tp53_panel_real, "k5_cosine_distance_to_WT", score_col, "local_k5"),
])

corr_table

In [ ]:
corr_table_round = corr_table.copy()

for col in corr_table_round.columns:
    if col not in ["label", "n"]:
        corr_table_round[col] = corr_table_round[col].round(6)

corr_table_round

In [ ]:
## BLOSUM

In [ ]:
from Bio.Align import substitution_matrices
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import spearmanr, pearsonr, kendalltau

blosum62 = substitution_matrices.load("BLOSUM62")


In [ ]:
def get_blosum_score(wt, mut, matrix):
    try:
        return -float(matrix[(wt, mut)])
    except Exception:
        try:
            return -float(matrix[(mut, wt)])
        except Exception:
            return np.nan

In [ ]:
tp53_panel_real = tp53_panel_real.copy()

tp53_panel_real["blosum62_score"] = [
    get_blosum_score(w, m, blosum62)
    for w, m in zip(tp53_panel_real["wt_aa"], tp53_panel_real["mut_aa"])
]

tp53_panel_real[["mutant", "wt_aa", "mut_aa", "blosum62_score"]].head(20)

In [ ]:
tp53_panel_real["combined_lof_dn_score"] = tp53_panel_real["score_raw"]
tp53_panel_real["activity_score"] = -tp53_panel_real["score_raw"]

tp53_panel_real[["mutant", "combined_lof_dn_score", "activity_score"]].head()

In [ ]:
def correlation_report(df, x_col, y_col, label=None):
    sub = df[[x_col, y_col]].dropna().copy()

    x = sub[x_col].values
    y = sub[y_col].values

    sp = spearmanr(x, y)
    pr = pearsonr(x, y)
    kt = kendalltau(x, y)

    return {
        "label": label if label is not None else x_col,
        "n": len(sub),
        "spearman_rho": sp.statistic,
        "spearman_p": sp.pvalue,
        "pearson_r": pr.statistic,
        "pearson_p": pr.pvalue,
        "kendall_tau": kt.statistic,
        "kendall_p": kt.pvalue,
    }

In [ ]:
corr_compare_deleterious = pd.DataFrame([
    correlation_report(tp53_panel_real, "blosum62_score", "combined_lof_dn_score", "BLOSUM62 vs deleteriousness"),
    correlation_report(tp53_panel_real, "k5_cosine_distance_to_WT", "combined_lof_dn_score", "ESM local k=5 vs deleteriousness"),
])

corr_compare_deleterious

In [ ]:
corr_compare_activity = pd.DataFrame([
    correlation_report(tp53_panel_real, "blosum62_score", "activity_score", "BLOSUM62 vs activity"),
    correlation_report(tp53_panel_real, "k5_cosine_distance_to_WT", "activity_score", "ESM local k=5 vs activity"),
])

corr_compare_activity

In [ ]:
x = tp53_panel_real["blosum62_score"].values
y = tp53_panel_real["combined_lof_dn_score"].values

rho, pval = spearmanr(x, y)

plt.figure(figsize=(7, 5))
plt.scatter(x, y, alpha=0.8)
plt.xlabel("BLOSUM62 score")
plt.ylabel("Combined LOF/DN score")
plt.title(f"BLOSUM62 vs TP53 deleteriousness-like score\nSpearman rho={rho:.3f}, p={pval:.3g}")
plt.show()

In [ ]:
x = tp53_panel_real["blosum62_score"].values
y = tp53_panel_real["k5_cosine_distance_to_WT"].values

rho, pval = spearmanr(x, y)

plt.figure(figsize=(7, 5))
plt.scatter(x, y, alpha=0.8)
plt.xlabel("BLOSUM62 score")
plt.ylabel("Local embedding shift (k=5)")
plt.title(f"BLOSUM62 vs ESM local k=5\nSpearman rho={rho:.3f}, p={pval:.3g}")
plt.show()

In [ ]:
tp53_panel_real["blosum_bin"] = pd.cut(
    tp53_panel_real["blosum62_score"],
    bins=[-10, -1, 1, 20],
    labels=["low/negative", "intermediate", "high"]
)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

tp53_panel_real.boxplot(column="combined_lof_dn_score", by="blosum_bin", grid=False, ax=axes[0])
axes[0].set_title("Deleteriousness by BLOSUM bin")
axes[0].set_xlabel("")
axes[0].set_ylabel("Combined LOF/DN score")

tp53_panel_real.boxplot(column="k5_cosine_distance_to_WT", by="blosum_bin", grid=False, ax=axes[1])
axes[1].set_title("ESM local shift by BLOSUM bin")
axes[1].set_xlabel("")
axes[1].set_ylabel("Local embedding shift (k=5)")

plt.suptitle("")
plt.tight_layout()
plt.show()

In [ ]:
benchmark_table = pd.DataFrame([
    correlation_report(tp53_panel_real, "blosum62_score", "combined_lof_dn_score", "BLOSUM62"),
    correlation_report(tp53_panel_real, "global_cosine_distance_to_WT", "combined_lof_dn_score", "ESM global"),
    correlation_report(tp53_panel_real, "k5_cosine_distance_to_WT", "combined_lof_dn_score", "ESM local k=5"),
])

benchmark_table

In [ ]:
tp53_panel_real.to_csv("tp53_panel_with_blosum.csv", index=False)
benchmark_table.to_csv("tp53_benchmark_blosum_vs_esm.csv", index=False)

print("Saved:")
print("- tp53_panel_with_blosum.csv")
print("- tp53_benchmark_blosum_vs_esm.csv")

In [ ]:
## Comparacion justa


df_cmp = tp53_panel_real.copy()

df_cmp["target"] = df_cmp["combined_lof_dn_score"]
df_cmp["esm_badness"] = df_cmp["k5_cosine_distance_to_WT"]
df_cmp["blosum_badness"] = -df_cmp["blosum62_score"]

df_cmp[[
    "mutant", "target", "esm_badness", "blosum62_score", "blosum_badness"
]].head()

In [ ]:
def subgroup_spearman(df, subset_name, min_n=10):
    sub = df[["target", "esm_badness", "blosum_badness"]].dropna().copy()
    n = len(sub)

    if n < min_n:
        return {
            "subset": subset_name,
            "n": n,
            "rho_esm": np.nan,
            "p_esm": np.nan,
            "rho_blosum": np.nan,
            "p_blosum": np.nan,
            "delta_rho_esm_minus_blosum": np.nan,
        }

    rho_esm, p_esm = spearmanr(sub["esm_badness"], sub["target"])
    rho_blo, p_blo = spearmanr(sub["blosum_badness"], sub["target"])

    return {
        "subset": subset_name,
        "n": n,
        "rho_esm": float(rho_esm),
        "p_esm": float(p_esm),
        "rho_blosum": float(rho_blo),
        "p_blosum": float(p_blo),
        "delta_rho_esm_minus_blosum": float(rho_esm - rho_blo),
    }

In [ ]:
global_cmp = pd.DataFrame([
    subgroup_spearman(df_cmp, "all_mutants", min_n=10)
])

global_cmp

In [ ]:
group_results = []

for g, subdf in df_cmp.groupby("group"):
    group_results.append(subgroup_spearman(subdf, f"group={g}", min_n=10))

group_results = pd.DataFrame(group_results).sort_values(
    "delta_rho_esm_minus_blosum", ascending=False
)

group_results

In [ ]:
flag_cols = [
    "class_change",
    "hydro_group_change",
    "is_conservative_like",
    "introduces_glycine",
    "introduces_proline",
    "introduces_cysteine",
]

flag_results = []

for col in flag_cols:
    for val in [True, False]:
        subdf = df_cmp[df_cmp[col] == val].copy()
        flag_results.append(subgroup_spearman(subdf, f"{col}={val}", min_n=10))

flag_results = pd.DataFrame(flag_results).sort_values(
    "delta_rho_esm_minus_blosum", ascending=False
)

flag_results

In [ ]:
df_cmp["blosum_bin"] = pd.cut(
    df_cmp["blosum62_score"],
    bins=[-10, -1, 1, 20],
    labels=["low_or_negative", "intermediate", "high"]
)

bin_results = []

for b, subdf in df_cmp.groupby("blosum_bin", observed=False):
    bin_results.append(subgroup_spearman(subdf, f"blosum_bin={b}", min_n=10))

bin_results = pd.DataFrame(bin_results).sort_values(
    "delta_rho_esm_minus_blosum", ascending=False
)

bin_results

In [ ]:
position_results = []

for pos, subdf in df_cmp.groupby("position"):
    wt = subdf["wt_aa"].iloc[0]
    position_results.append(subgroup_spearman(subdf, f"{wt}{pos}", min_n=8))

position_results = pd.DataFrame(position_results).sort_values(
    "delta_rho_esm_minus_blosum", ascending=False
)

position_results

In [ ]:
all_subgroup_results = pd.concat(
    [
        global_cmp.assign(kind="global"),
        group_results.assign(kind="group"),
        flag_results.assign(kind="flag"),
        bin_results.assign(kind="blosum_bin"),
        position_results.assign(kind="position"),
    ],
    ignore_index=True
)

all_subgroup_results.sort_values("delta_rho_esm_minus_blosum", ascending=False).head(30)

In [ ]:
plot_df = all_subgroup_results.copy()
plot_df = plot_df.dropna(subset=["delta_rho_esm_minus_blosum"])
plot_df = plot_df.sort_values("delta_rho_esm_minus_blosum", ascending=True)

plt.figure(figsize=(10, 12))
plt.barh(plot_df["subset"], plot_df["delta_rho_esm_minus_blosum"])
plt.axvline(0, linestyle="--")
plt.xlabel("Spearman rho(ESM, target) - rho(BLOSUM, target)")
plt.ylabel("Subset")
plt.title("Where does ESM win over BLOSUM?")
plt.show()

In [ ]:
reliable_subsets = all_subgroup_results[
    all_subgroup_results["n"] >= 12
].sort_values("delta_rho_esm_minus_blosum", ascending=False)

reliable_subsets

In [ ]:
def bootstrap_delta_spearman(df, n_boot=2000, random_state=123):
    rng = np.random.default_rng(random_state)

    sub = df[["target", "esm_badness", "blosum_badness"]].dropna().copy()
    n = len(sub)

    if n < 10:
        return {
            "n": n,
            "delta_rho": np.nan,
            "ci_low": np.nan,
            "ci_high": np.nan,
        }

    # delta observado
    rho_esm, _ = spearmanr(sub["esm_badness"], sub["target"])
    rho_blo, _ = spearmanr(sub["blosum_badness"], sub["target"])
    delta_obs = rho_esm - rho_blo

    boot = []
    idx = np.arange(n)

    for _ in range(n_boot):
        samp = rng.choice(idx, size=n, replace=True)
        s = sub.iloc[samp]
        r1, _ = spearmanr(s["esm_badness"], s["target"])
        r2, _ = spearmanr(s["blosum_badness"], s["target"])
        boot.append(r1 - r2)

    ci_low, ci_high = np.percentile(boot, [2.5, 97.5])

    return {
        "n": n,
        "delta_rho": float(delta_obs),
        "ci_low": float(ci_low),
        "ci_high": float(ci_high),
    }

In [ ]:
bootstrap_rows = []

# global
res = bootstrap_delta_spearman(df_cmp, n_boot=2000)
bootstrap_rows.append({"subset": "all_mutants", **res})

# grupos
for g, subdf in df_cmp.groupby("group"):
    res = bootstrap_delta_spearman(subdf, n_boot=2000)
    bootstrap_rows.append({"subset": f"group={g}", **res})

# flags
for col in [
    "class_change",
    "is_conservative_like",
    "introduces_glycine",
    "introduces_proline",
    "introduces_cysteine",
]:
    for val in [True, False]:
        subdf = df_cmp[df_cmp[col] == val]
        res = bootstrap_delta_spearman(subdf, n_boot=2000)
        bootstrap_rows.append({"subset": f"{col}={val}", **res})

bootstrap_df = pd.DataFrame(bootstrap_rows).sort_values("delta_rho", ascending=False)
bootstrap_df

In [ ]:
tmp = bootstrap_df.dropna(subset=["delta_rho"]).sort_values("delta_rho", ascending=True)

plt.figure(figsize=(10, 8))
y = np.arange(len(tmp))

plt.errorbar(
    x=tmp["delta_rho"],
    y=y,
    xerr=[
        tmp["delta_rho"] - tmp["ci_low"],
        tmp["ci_high"] - tmp["delta_rho"]
    ],
    fmt="o"
)

plt.yticks(y, tmp["subset"])
plt.axvline(0, linestyle="--")
plt.xlabel("Delta Spearman rho (ESM - BLOSUM)")
plt.title("Where does ESM outperform BLOSUM? Bootstrap CI")
plt.show()

In [ ]:
all_subgroup_results.to_csv("tp53_esm_vs_blosum_subgroup_correlations.csv", index=False)
bootstrap_df.to_csv("tp53_esm_vs_blosum_bootstrap_delta_rho.csv", index=False)

print("Saved:")
print("- tp53_esm_vs_blosum_subgroup_correlations.csv")
print("- tp53_esm_vs_blosum_bootstrap_delta_rho.csv")

In [ ]:
### Figura final

In [ ]:

final_subsets = [
    "all_mutants",
    "group=comparison_dbd",
    "group=hotspot",
]

final_df = bootstrap_df[bootstrap_df["subset"].isin(final_subsets)].copy()

order_map = {
    "all_mutants": 0,
    "group=comparison_dbd": 1,
    "group=hotspot": 2,
}

label_map = {
    "all_mutants": "All mutants",
    "group=comparison_dbd": "Comparison DBD",
    "group=hotspot": "Hotspots",
}

final_df["order"] = final_df["subset"].map(order_map)
final_df["label"] = final_df["subset"].map(label_map)
final_df = final_df.sort_values("order").reset_index(drop=True)

final_df

In [ ]:
plt.figure(figsize=(7, 4.8))

y = np.arange(len(final_df))

x = final_df["delta_rho"].values
xerr_left = x - final_df["ci_low"].values
xerr_right = final_df["ci_high"].values - x

plt.errorbar(
    x=x,
    y=y,
    xerr=[xerr_left, xerr_right],
    fmt="o",
    capsize=4,
    markersize=7,
    linewidth=1.5,
)

plt.axvline(0, linestyle="--", linewidth=1)

plt.yticks(y, final_df["label"])
plt.xlabel(r"$\Delta \rho = \rho_{\mathrm{ESM}} - \rho_{\mathrm{BLOSUM}}$")
plt.title("Relative advantage of ESM over BLOSUM across TP53 mutant subsets")

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(7.5, 5))

y = np.arange(len(final_df))
x = final_df["delta_rho"].values
xerr_left = x - final_df["ci_low"].values
xerr_right = final_df["ci_high"].values - x

plt.errorbar(
    x=x,
    y=y,
    xerr=[xerr_left, xerr_right],
    fmt="o",
    capsize=4,
    markersize=7,
    linewidth=1.5,
)

plt.axvline(0, linestyle="--", linewidth=1)

for i, row in final_df.iterrows():
    txt = f"{row['delta_rho']:.3f} [{row['ci_low']:.3f}, {row['ci_high']:.3f}]"
    plt.text(row["ci_high"] + 0.01, i, txt, va="center", fontsize=9)

plt.yticks(y, final_df["label"])
plt.xlabel(r"$\Delta \rho = \rho_{\mathrm{ESM}} - \rho_{\mathrm{BLOSUM}}$")
plt.title("Bootstrap comparison of ESM and BLOSUM across selected TP53 subsets")

plt.tight_layout()
plt.show()

In [ ]:
final_table = final_df[["label", "n", "delta_rho", "ci_low", "ci_high"]].copy()

for col in ["delta_rho", "ci_low", "ci_high"]:
    final_table[col] = final_table[col].round(3)

final_table

In [ ]:
final_table.to_csv("esm_vs_BLOSUMboots_group.csv", index=False)

print("Saved: tp53_esm_vs_blosum_selected_subsets_table.csv")

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 5))

y = np.arange(len(final_df))
x = final_df["delta_rho"].values
xerr_left = x - final_df["ci_low"].values
xerr_right = final_df["ci_high"].values - x

ax.errorbar(
    x=x,
    y=y,
    xerr=[xerr_left, xerr_right],
    fmt="o",
    capsize=4,
    markersize=7,
    linewidth=1.5,
)

ax.axvline(0, linestyle="--", linewidth=1)

for i, row in final_df.iterrows():
    txt = f"{row['delta_rho']:.3f} [{row['ci_low']:.3f}, {row['ci_high']:.3f}]"
    ax.text(row["ci_high"] + 0.01, i, txt, va="center", fontsize=9)

ax.set_yticks(y)
ax.set_yticklabels(final_df["label"])
ax.set_xlabel(r"$\Delta \rho = \rho_{\mathrm{ESM}} - \rho_{\mathrm{BLOSUM}}$")
ax.set_title("Comparacion de ESM vs BLOSUM para grupos de mutaciones")

plt.tight_layout()
plt.savefig("tp53_esm_vs_blosum_selected_subsets.svg")
plt.show()

print("Saved: tp53_esm_vs_blosum_selected_subsets.png")